In [ ]:
import pandas as pd
import numpy as np
import numpyro
from jax import numpy as jnp
from jax.random import PRNGKey
from numpyro import distributions as dist
from numpyro import infer
import matplotlib.pyplot as plt
import diffrax as dfx

from summer3.epi import CompartmentalModelODE, CategoryData, strat_data_from_pandas, build_istate, dti_to_epoch

from tb_macro.constants import ALL_COMPARTMENTS, AGE_STRATA, ISO3, START_TIME, END_TIME, YOUNG_END_AGE
from tb_macro.epi import get_base_model, add_natural_history, add_seeding, add_latency_flows, add_infection_flows
from tb_macro.inputs import load_demography, load_fertility, load_who_outcomes
from tb_macro.parameters import BASE_PARAMS
from tb_macro.demography import add_replacement_deaths, add_ageing_flows, prepare_pop_data_for_entries, add_entry_births
from tb_macro.health_system import add_treatment_flows, add_detection
from tb_macro.calibration import make_log_likelihood
from tb_macro.targets import NOTIF_TARGET, LATENT_TARGET

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Model construction
group_popsize, death_rates, age_weights = load_demography(ISO3)
fert_padded = load_fertility(ISO3)
tsr, death_in_unsucc, who_mort = load_who_outcomes(ISO3)
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))

add_infection_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat, age_weights, group_popsize, fert_padded, YOUNG_END_AGE, START_TIME)
add_natural_history(epi_model, disease_state, age_strat, clin_strat, infect_strat)
add_ageing_flows(epi_model, age_strat)
add_seeding(epi_model, disease_state, START_TIME)
add_detection(epi_model, disease_state, clin_strat, START_TIME)
add_replacement_deaths(epi_model, disease_state, age_strat, death_rates, START_TIME)
add_entry_births(epi_model, disease_state, age_strat, START_TIME, entry_rates, entry_times)
add_treatment_flows(death_rates, START_TIME, epi_model, disease_state, age_strat, infect_strat, clin_strat, tsr, death_in_unsucc)
add_latency_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat)

# Initialisation
init_apops_series = pd.Series(index=[str(a) for a in AGE_STRATA], data=np.array(start_apops))
init_apops = strat_data_from_pandas(init_apops_series, age_strat)
init_dpops = [0.0] * len(ALL_COMPARTMENTS)
init_dpops[ALL_COMPARTMENTS.index("mtb_naive")] = 1.0
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_dpops)))]
epi_model.set_initial_population(init_apops, pop_splits)
epi_model.computed_values.append("dynamic_mm")

In [ ]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

runner, istate = get_runner(epi_model)

In [ ]:
solver_kwargs = {
    "max_steps": 4000,
    "stepsize_controller": dfx.PIDController(rtol=1e-5, atol=1e-5, dtmax=7.0),
    "adjoint": dfx.RecursiveCheckpointAdjoint(2048),
}

In [ ]:
latent_date = LATENT_TARGET.index[0]
latent_target_val = LATENT_TARGET.iloc[0] / 1e2

calib_params = ["contact_rate", "detect_val_2"]

log_like = make_log_likelihood(epi_model, disease_state, solver_kwargs, latent_date, latent_target_val, NOTIF_TARGET, who_mort)

In [ ]:
# Calibration
priors = {
    "contact_rate": dist.Uniform(5.0, 11.0),
    "detect_val_2": dist.Uniform(0.4, 0.7),
}


def model():
    params = BASE_PARAMS | {k: numpyro.sample(k, v) for k, v in priors.items()}
    ll = log_like(params)
    numpyro.factor("ll", ll)


# kernel = infer.SA(model)  # , adapt_state_size=4)  # infer.NUTS(model, max_tree_depth=5)
kernel = infer.NUTS(model, max_tree_depth=5, init_strategy=infer.init_to_median())
mcmc = infer.MCMC(kernel, num_warmup=100, num_samples=100, num_chains=4)
mcmc.run(PRNGKey(2))